In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import scipy
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from itertools import product
import json, joblib, torch



# NVIDIA GPU → Apple MPS → CPU 순으로 자동 감지
def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

TORCH_DEVICE = get_device()

# PyMC(PyTensor) / JAX 모두 Apple MPS 미지원
# NVIDIA → JAX 가속 "numpyro"
# Apple Silicon / CPU → "nuts" (CPU 전용)
MCMC_SAMPLER = "numpyro" if torch.cuda.is_available() else "nuts"

print(f"[PyTorch device] {TORCH_DEVICE}")
print(f"[MCMC sampler ] {MCMC_SAMPLER}")

## 1. 데이터 로드

In [ ]:
df = pd.read_csv("/Users/ijongseung/Demadn-Quadratic-Tilting/power_demand_final.csv")
df["일시"] = pd.to_datetime(df["일시"])
df["holiday_name"].fillna("non-event", inplace=True)

## 2. 추세 추출 — HP-filter

논문 Algorithm 1, Step A1

In [ ]:
import numpy as np
from statsmodels.tsa.filters.hp_filter import hpfilter

# ── 추세 추출: HP-filter (Hodrick-Prescott filter) ──────────────────────────
# 논문 Algorithm 1 Step A1 권장 기법: STL / HP-filter / 이동평균
#
# ※ HP-filter endpoint problem:
#   HP-filter는 양방향(two-sided) 필터이므로 데이터 양 끝단에서
#   추세가 실제값 쪽으로 왜곡(끌림)되는 현상이 발생한다.
#   이를 방지하기 위해 전체 데이터(train+val+test)에 적용한다.
#   추세는 예측 대상이 아닌 구조적 분해 성분이므로
#   테스트 구간 포함이 데이터 누출(leakage)에 해당하지 않는다.
#
# λ 설정 (Ravn & Uhlig 2002):
#   λ_h = 1600 × (8766/4)^4 ≈ 1.28e11
#   실용 범위에서 장기 추세만 포착 → 1.28e8 사용
LAMBDA_HP = 1.28e8

# 학습 분할 기준 (Cell 4 이후에서도 사용)
train = df[df["일시"] <= "2022-12-31 23:00:00"]

# 전체 데이터에 HP-filter 적용 (endpoint 왜곡 제거)
y_all_raw = df["power demand(MW)"].values
_, trend_all = hpfilter(y_all_raw, lamb=LAMBDA_HP)   # (N_all,)

# 데이터프레임에 저장
df["trend"]   = trend_all
df["detrend"] = df["power demand(MW)"] - df["trend"]   # 추세 제거 (원 단위 MW)

print(f"Trend range : {trend_all.min():.1f} ~ {trend_all.max():.1f} MW")
print(f"Detrend std : {df['detrend'].std():.1f} MW")


## 3. Fourier 계절성 분해

논문 Algorithm 1, Step A2

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from itertools import product

# 대상: detrend (원 단위 MW, 선형 공간)
y_all = df["detrend"].values
t_all = np.arange(len(y_all))

# Fourier term 생성
def generate_fourier_terms(timesteps, period, K):
    terms = []
    for k in range(1, K + 1):
        terms.append(np.sin(2 * np.pi * k * timesteps / period))
        terms.append(np.cos(2 * np.pi * k * timesteps / period))
    return np.stack(terms, axis=1)

# 탐색할 K 값
daily_K_vals  = [1,2,3]
weekly_K_vals = [1,2,3,4,5,6,7,8]
yearly_K_vals = [1,2,3,4]

best_mse = float("inf")
best_model, best_X_all, best_K_combo = None, None, None

# 학습/검증 분할
val = df[(df["일시"] > "2022-12-31 23:00:00") & (df["일시"] <= "2023-12-31 23:00:00")]
train_end = len(train)
val_end   = train_end + len(val)

for Kd, Kw, Ky in product(daily_K_vals, weekly_K_vals, yearly_K_vals):
    X_all = np.hstack([
        generate_fourier_terms(t_all, 24, Kd),          # 1일
        generate_fourier_terms(t_all, 24*7, Kw),        # 1주
        generate_fourier_terms(t_all, 24*365.25, Ky)    # 1년
    ])
    X_train = X_all[:train_end]
    X_val   = X_all[train_end:val_end]
    y_train = y_all[:train_end]
    y_val   = y_all[train_end:val_end]

    model = LinearRegression().fit(X_train, y_train)
    val_pred = model.predict(X_val)
    mse = mean_squared_error(y_val, val_pred)

    if mse < best_mse:
        best_mse = mse
        best_model = model
        best_X_all = X_all
        best_K_combo = (Kd, Kw, Ky)

# 최적 모델로 전체 예측 (원 단위 MW)
seasonality_pred = best_model.predict(best_X_all)

df["seasonality"] = seasonality_pred


### 4. Fourier 성분 시각화 및 분리

In [ ]:
# --- 1. 시간 인덱스 보정 ---
if "t" not in df.columns:
    df["t"] = np.arange(len(df))

df_result = df.copy()

# --- 2. 최적 K ---
Kd, Kw, Ky = best_K_combo

# --- 3. Fourier term 생성 (원 단위 MW) ---
daily_terms  = generate_fourier_terms(t_all, 24, Kd)
weekly_terms = generate_fourier_terms(t_all, 24*7, Kw)
yearly_terms = generate_fourier_terms(t_all, 24*365.25, Ky)

coef = best_model.coef_

# --- 4. 각 성분 분리 (원 단위 MW) ---
daily_pred  = daily_terms  @ coef[:2*Kd]
weekly_pred = weekly_terms @ coef[2*Kd:2*Kd + 2*Kw]
yearly_pred = yearly_terms @ coef[2*Kd + 2*Kw:]

# --- 5. 전체 seasonality & 잔차 (원 단위 MW) ---
seasonality_pred = best_model.predict(best_X_all)
residuals = y_all - seasonality_pred   # r_t^(0) = D_t - T̂_t - Ŝ_t^(F)

df_result["Fourier_Seasonality"] = seasonality_pred
df_result["Fourier_Residual"]    = residuals

# --- 6. 시각화 ---
plt.figure(figsize=(16, 12))

plt.subplot(5, 1, 1)
plt.plot(df_result["t"][:24*31], daily_pred[:24*31], label="Daily Seasonality (MW)", color="blue")
plt.title("Daily Seasonality Component (MW)")
plt.grid(True); plt.legend()

plt.subplot(5, 1, 2)
plt.plot(df_result["t"][:24*31], weekly_pred[:24*31], label="Weekly Seasonality (MW)", color="orange")
plt.title("Weekly Seasonality Component (MW)")
plt.grid(True); plt.legend()

plt.subplot(5, 1, 3)
plt.plot(df_result["t"], yearly_pred, label="Yearly Seasonality (MW)", color="green")
plt.title("Yearly Seasonality Component (MW)")
plt.grid(True); plt.legend()

plt.subplot(5, 1, 4)
plt.plot(df_result["t"], df_result["Fourier_Seasonality"], label="Total Fourier Seasonality (MW)", color="purple")
plt.title("Fourier Seasonality — Total (MW)")
plt.grid(True); plt.legend()

plt.subplot(5, 1, 5)
plt.plot(df_result["t"], df_result["Fourier_Residual"], label="Residuals (MW)", color="red")
plt.title("Fourier Residuals r_t^(0) (MW)")
plt.grid(True); plt.legend()

plt.tight_layout()
plt.show()


### 5. 진폭·위상 분석 (선택)

In [ ]:
def extract_amplitude_phase(coef_block, K, label):
    results = []
    for k in range(K):
        a_k = coef_block[2 * k]
        b_k = coef_block[2 * k + 1]
        amplitude = np.sqrt(a_k**2 + b_k**2)
        phase_rad = np.arctan2(b_k, a_k)
        phase_deg = np.degrees(phase_rad)
        results.append({
            "Component": label,
            "Harmonic k": k + 1,
            "aₖ": round(a_k, 4),
            "bₖ": round(b_k, 4),
            "Amplitude": round(amplitude, 4),
            "Phase (rad)": round(phase_rad, 4),
            "Phase (deg)": round(phase_deg, 2)
        })
    return results

# 최적 K
Kd, Kw, Ky = best_K_combo
coef = best_model.coef_

# 계수 분할
daily_coef   = coef[:2 * Kd]
weekly_coef  = coef[2 * Kd : 2 * Kd + 2 * Kw]
yearly_coef  = coef[2 * Kd + 2 * Kw:]

# 해석 실행
daily_info  = extract_amplitude_phase(daily_coef,  Kd, "Daily")
weekly_info = extract_amplitude_phase(weekly_coef, Kw, "Weekly")
yearly_info = extract_amplitude_phase(yearly_coef, Ky, "Yearly")

# 결과 통합
seasonality_df = pd.DataFrame(daily_info + weekly_info + yearly_info)

# 시각 또는 테이블 출력

print(seasonality_df.to_string(index=False))

# 누적 설명력 계산

def confidential_test(data, component):
    # 필터링 
    data = data[data["Component"]==component]
    energy = data["aₖ"].values**2 + data["bₖ"].values**2

    total_energy = np.sum(energy)
    cumulative_ratio = np.cumsum(energy) / total_energy

    print("누적 설명력:", np.round(cumulative_ratio, 4))

    return np.round(cumulative_ratio, 4)

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np

# 각각의 누적 설명력 시리즈 생성
daily = confidential_test(seasonality_df, "Daily")
weekly = confidential_test(seasonality_df, "Weekly")
yearly = confidential_test(seasonality_df, "Yearly")

# Subplot 설정 (1행 3열)
fig, axes = plt.subplots(1, 3, figsize=(10, 5), sharex=False)

# 공통: x축 눈금을 정수로 고정
for ax in axes:
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

# 1. Daily
axes[0].plot(np.arange(1, len(daily)+1), daily, marker='o', color='blue')
axes[0].axhline(y=0.95, color='r', linestyle='--', label='95% Cumulative')
axes[0].set_title("Daily CEVR")
axes[0].set_ylabel("Cumulative Ratio")
axes[0].legend()

# 2. Weekly
axes[1].plot(np.arange(1, len(weekly)+1), weekly, marker='o', color='orange')
axes[1].axhline(y=0.95, color='r', linestyle='--', label='95% Cumulative')
axes[1].set_title("Weekly CEVR")
axes[1].set_ylabel("Cumulative Ratio")
axes[1].legend()

# 3. Yearly
axes[2].plot(np.arange(1, len(yearly)+1), yearly, marker='o', color='green')
axes[2].axhline(y=0.95, color='r', linestyle='--', label='95% Cumulative')
axes[2].set_title("Yearly CEVR")
axes[2].set_xlabel("Harmonic k")
axes[2].set_ylabel("Cumulative Ratio")
axes[2].legend()

plt.tight_layout()
plt.show()


## 6. Train / Val / Test 분할

In [ ]:
train = df_result[df_result["일시"]<="2022-12-31 23:00:00"]
val = df_result[(df_result["일시"]>"2022-12-31 23:00:00") & (df["일시"]<="2023-12-31 23:00:00")]
test = df_result[df_result["일시"]>"2023-12-31 23:00:00"]

## 7. Seq2Seq LSTM — 모델 정의

논문 Algorithm 1, Step A4

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from datetime import datetime
from typing import Optional

# =========================================================
# 1) Dataset (일 단위 stride, 패딩+마스크, 빈 타깃 제외)
# =========================================================
class DailyStepDataset(Dataset):
    """
    - in_days: 입력 일 수 (정수)
    - out_days: 예측 일 수 (정수)
    - stride: 하루(24h) 고정 이동
    - feature_cols: 모델 입력에 쓸 모든 피처 (X)
    - target_col: 예측 대상 단일 컬럼명 (y)
    - scaler_X, scaler_y: 반드시 train으로만 fit해서 전달
    - pad_strategy: "zero" | "edge"
    - include_head_tail: True면 시계열 헤드/테일도 윈도우 생성(패딩으로 보정)
    - exclude_empty_targets: 타깃 마스크 합이 0인 샘플 제거
    """
    def __init__(self, df, feature_cols, target_col,
                 in_days, out_days, scaler_X, scaler_y,
                 pad_value=0.0, pad_strategy="zero",
                 include_head_tail=False,
                 exclude_empty_targets=True):
                 
        self.feature_cols = feature_cols
        self.target_col = target_col
        self.in_len = int(in_days) * 24
        self.out_len = int(out_days) * 24
        self.stride = 24

        Xs = scaler_X.transform(df[feature_cols].values)
        ys = scaler_y.transform(df[[target_col]].values).flatten()
        self.X = torch.tensor(Xs, dtype=torch.float32)
        self.y = torch.tensor(ys, dtype=torch.float32)

        self.T = len(self.X)
        self.pad_value = float(pad_value)
        self.pad_strategy = pad_strategy

        if include_head_tail:
            start_min = -(self.in_len - 1)
            start_max = self.T - 1
        else:
            start_min = 0
            start_max = self.T - self.in_len - self.out_len
        starts = list(range(start_min, start_max + 1, self.stride))

        # 필요 시 빈 타깃 샘플 제거
        if exclude_empty_targets:
            filtered = []
            for s in starts:
                out_start, out_end = s + self.in_len, s + self.in_len + self.out_len - 1
                real_out_s, real_out_e = max(out_start, 0), min(out_end, self.T - 1)
                y_real_len = max(0, real_out_e - real_out_s + 1)
                if y_real_len > 0:
                    filtered.append(s)
            self.starts = filtered
        else:
            self.starts = starts

    def __len__(self):
        return len(self.starts)

    def _left_pad(self, arr, need, feat=False):
        if need <= 0:
            return arr
        if self.pad_strategy == "edge" and len(arr) > 0:
            pad_val = arr[0:1] if not feat else arr[0:1, :]
            pad = pad_val.repeat(need, 1) if feat else pad_val.repeat(need)
        else:
            shape = (need, arr.shape[1]) if feat else (need,)
            pad = torch.full(shape, self.pad_value, dtype=arr.dtype)
        return torch.cat([pad, arr], dim=0)

    def _right_pad(self, arr, need):
        if need <= 0:
            return arr
        if self.pad_strategy == "edge" and len(arr) > 0:
            pad_val = arr[-1:]
            pad = pad_val.repeat(need)
        else:
            pad = torch.full((need,), self.pad_value, dtype=arr.dtype)
        return torch.cat([arr, pad], dim=0)

    def __getitem__(self, idx):
        s = self.starts[idx]
        in_start, in_end = s, s + self.in_len - 1
        out_start, out_end = s + self.in_len, s + self.in_len + self.out_len - 1

        # 입력 X 윈도우 (좌패딩 가능)
        real_in_s, real_in_e = max(in_start, 0), min(in_end, self.T - 1)
        x_part = self.X[real_in_s:real_in_e+1]
        left_need = real_in_s - in_start
        x_seq = self._left_pad(x_part, left_need, feat=True)

        # 타깃 y 윈도우 (우패딩/좌패딩 모두 가능)
        real_out_s, real_out_e = max(out_start, 0), min(out_end, self.T - 1)
        y_part = self.y[real_out_s:real_out_e+1] if real_out_s <= real_out_e else torch.empty(0)
        right_need = out_end - real_out_e
        y_seq = self._right_pad(y_part, right_need)
        if len(y_seq) < self.out_len:  # 헤드 구간은 좌패딩 필요할 수 있음
            y_seq = self._left_pad(y_seq, self.out_len - len(y_seq), feat=False)

        # 마스크: 실제 관측된 타깃 위치 = 1.0
        y_mask = torch.zeros(self.out_len, dtype=torch.float32)
        y_real_len = max(0, real_out_e - real_out_s + 1)
        if y_real_len > 0:
            left_pad_len = max(0, real_out_s - out_start)
            y_mask[left_pad_len:left_pad_len + y_real_len] = 1.0

        # 디코더 시작용 y0_prev (인코더 마지막 시점의 실제값 또는 pad_value)
        y0_idx = in_end
        if 0 <= y0_idx < self.T:
            y0_prev = self.y[y0_idx]
        else:
            y0_prev = torch.tensor(self.pad_value, dtype=self.y.dtype)

        return x_seq, y_seq.unsqueeze(-1), y_mask, y0_prev


# =========================================================
# 2) 모델 (Seq2Seq; 패딩-안전 Teacher Forcing)
# =========================================================
class Encoder(nn.Module):
    def __init__(self, input_dim, hid_dim, n_layers=1, dropout=0.0):
        super().__init__()
        self.lstm = nn.LSTM(
            input_dim, hid_dim, n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0
        )
    def forward(self, x):
        return self.lstm(x)  # enc_outputs, (h,c)

class Decoder(nn.Module):
    def __init__(self, input_dim, hid_dim, out_len, n_layers=1, dropout=0.0):
        super().__init__()
        assert input_dim == 1, "Decoder input must be scalar y (1)."
        self.out_len = out_len
        self.lstm = nn.LSTM(
            input_dim, hid_dim, n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0
        )
        self.fc = nn.Linear(hid_dim, 1)

    def forward(self, enc_outputs, states, y0, y_target=None, y_mask=None, teacher_forcing_ratio=0.5):
        h, c = states
        B = h.size(1)
        device = h.device
        inp = y0.unsqueeze(1).unsqueeze(2)  # [B,1,1]
        outputs = []
        for t in range(self.out_len):
            out, (h, c) = self.lstm(inp, (h, c))        # out: [B,1,hid]
            pred = self.fc(out.squeeze(1)).unsqueeze(1) # [B,1,1]
            outputs.append(pred)

            # 패딩 구간엔 TF 금지, 나머지는 확률적으로 TF
            if (y_target is not None) and (y_mask is not None):
                valid = (y_mask[:, t] > 0.5).float()              # [B]
                if teacher_forcing_ratio > 0.0 and valid.sum() > 0:
                    rand = torch.rand(B, device=device)
                    use_tf = (rand < teacher_forcing_ratio) * valid  # [B]
                    y_t = y_target[:, t].unsqueeze(1).unsqueeze(2)   # [B,1,1]
                    use_tf_bool = use_tf.bool().view(B,1,1)
                    inp = torch.where(use_tf_bool, y_t, pred)
                else:
                    inp = pred
            else:
                inp = pred
        return torch.cat(outputs, dim=1)  # [B,out_len,1]

class Seq2Seq(nn.Module):
    def __init__(self, enc_input_dim, dec_input_dim, hid_dim,
                 out_len, n_layers=1, dropout=0.0):
        super().__init__()
        self.encoder = Encoder(enc_input_dim, hid_dim, n_layers, dropout)
        self.decoder = Decoder(dec_input_dim, hid_dim, out_len, n_layers, dropout)
    def forward(self, x, y0, y_target=None, y_mask=None, teacher_forcing_ratio=0.5):
        enc_out, (h, c) = self.encoder(x)
        return self.decoder(enc_out, (h, c), y0,
                            y_target=y_target, y_mask=y_mask,
                            teacher_forcing_ratio=teacher_forcing_ratio)


# =========================================================
# 3) 학습/평가 루틴 (마스크 손실, ES, grad clip, 역스케일 지표)
# =========================================================

def masked_mse(preds, targets, mask, eps=1e-8):
    # preds/targets: [B, out_len] or [B, out_len, 1]; mask: [B, out_len]
    if preds.ndim == 3:
        preds = preds.squeeze(-1)
    if targets.ndim == 3:
        targets = targets.squeeze(-1)
    se = (preds - targets) ** 2
    se = se * mask
    denom = torch.clamp(mask.sum(), min=eps)
    return se.sum() / denom


def train_model(
    X_train, Y_train, M_train, y0_train,
    X_val,   Y_val,   M_val,   y0_val,
    X_test,  Y_test,  M_test,  y0_test,
    scaler_y,
    best_params: dict,
    device: torch.device,
    max_epochs: int = 150,
    patience: int = 10,
    grad_clip: Optional[float] = 1.0,
    ckpt_path: Optional[str] = None
):
    # Y shape 보정
    def sqz(y):
        return y.squeeze(-1) if (y.ndim == 3 and y.shape[-1] == 1) else y
    Y_train, Y_val, Y_test = sqz(Y_train), sqz(Y_val), sqz(Y_test)
    out_len = Y_train.shape[-1]

    # 모델
    model = Seq2Seq(
        enc_input_dim=X_train.shape[-1],
        dec_input_dim=1,
        hid_dim=int(best_params['units']),
        out_len=out_len,
        n_layers=int(best_params['lstm_layers']),
        dropout=float(best_params.get('lstm_dropout_rate', 0.0))
    ).to(device)

    criterion = masked_mse
    optimizer = optim.AdamW(model.parameters(), lr=float(best_params['learning_rate']))
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    # Tensor 변환
    def to_tensor_np(*arrs):
        return [torch.from_numpy(a).float() for a in arrs]
    Xtr, Ytr, Mtr, y0tr = to_tensor_np(X_train, Y_train, M_train, y0_train)
    Xva, Yva, Mva, y0va = to_tensor_np(X_val,   Y_val,   M_val,   y0_val)
    Xte, Yte, Mte, y0te = to_tensor_np(X_test,  Y_test,  M_test,  y0_test)

    tr_loader = DataLoader(TensorDataset(Xtr, Ytr, Mtr, y0tr), batch_size=int(best_params['batch_size']))
    va_loader = DataLoader(TensorDataset(Xva, Yva, Mva, y0va), batch_size=int(best_params['batch_size']))
    te_loader = DataLoader(TensorDataset(Xte, Yte, Mte, y0te), batch_size=int(best_params['batch_size']))

    # 학습 루프
    best_val = float('inf')
    bad = 0
    train_losses, val_losses = [], []

    if ckpt_path is None:
        stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        ckpt_path = f'best_model_{stamp}.pth'

    tfr = float(best_params.get('teacher_forcing_ratio', 0.5))

    for epoch in range(1, max_epochs + 1):
        # ---- Train ----
        model.train()
        total_tr = 0.0
        for xb, yb, mb, y0b in tr_loader:
            xb, yb, mb, y0b = xb.to(device), yb.to(device), mb.to(device), y0b.to(device)
            optimizer.zero_grad(set_to_none=True)
            preds = model(xb, y0b, y_target=yb, y_mask=mb, teacher_forcing_ratio=tfr)
            loss = criterion(preds, yb, mb)
            loss.backward()
            if grad_clip is not None:
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
            total_tr += loss.item()
        tr_loss = total_tr / max(1, len(tr_loader))
        train_losses.append(tr_loss)

        # ---- Validate ----
        model.eval()
        total_va = 0.0
        with torch.no_grad():
            for xb, yb, mb, y0b in va_loader:
                xb, yb, mb, y0b = xb.to(device), yb.to(device), mb.to(device), y0b.to(device)
                preds = model(xb, y0b, y_target=None, y_mask=None, teacher_forcing_ratio=0.0)
                total_va += criterion(preds, yb, mb).item()
        val_loss = total_va / max(1, len(va_loader))
        val_losses.append(val_loss)
        scheduler.step(val_loss)

        # ---- Early stopping ----
        if val_loss < best_val - 1e-8:
            best_val = val_loss
            bad = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            bad += 1

        if epoch == 1 or epoch % 10 == 0:
            lr_now = optimizer.param_groups[0]['lr']
            print(f"[{epoch}/{max_epochs}] Train={tr_loss:.4f}  Val={val_loss:.4f}  LR={lr_now:.2e}  (bad={bad}/{patience})")

        if epoch > 20 and bad >= patience:
            print(f"Early stopping at epoch {epoch} (best_val={best_val:.6f})")
            break

    # ---- 공통 추론 유틸: loader 단위 예측/지표 산출 ----
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()
    def _infer_and_metrics(loader, Y_true_np, M_np):
        preds_list = []
        masks_list = []
        with torch.no_grad():
            for xb, yb, mb, y0b in loader:
                xb, yb, mb, y0b = xb.to(device), yb.to(device), mb.to(device), y0b.to(device)
                preds = model(xb, y0b, y_target=None, y_mask=None, teacher_forcing_ratio=0.0)
                preds_list.append(preds.cpu())
                masks_list.append(mb.cpu())
        preds_all = torch.cat(preds_list, dim=0).squeeze(-1).numpy()
        masks_all = torch.cat(masks_list, dim=0).numpy().astype(bool)

        y_pred_inv = scaler_y.inverse_transform(preds_all.reshape(-1, 1)).reshape(preds_all.shape)
        y_true_inv = scaler_y.inverse_transform(Y_true_np.reshape(-1, 1)).reshape(Y_true_np.shape)

        diff = (y_pred_inv - y_true_inv)
        mask = masks_all
        denom = mask.sum()
        if denom == 0:
            metrics = {"mae": float('nan'), "rmse": float('nan'), "mape": float('nan')}
        else:
            mae  = float(np.abs(diff)[mask].mean())
            rmse = float(np.sqrt((diff[mask] ** 2).mean()))
            mape = float((np.abs(diff[mask]) / np.clip(np.abs(y_true_inv[mask]), 1e-8, None)).mean() * 100.0)
            metrics = {"mae": mae, "rmse": rmse, "mape": mape}
        return y_pred_inv, masks_all, metrics


    # ---- Test ----
    test_preds_inv, test_masks, test_metrics = _infer_and_metrics(te_loader, Y_test, M_test)
    print(f"[TEST]  MAE={test_metrics['mae']:.3f}  RMSE={test_metrics['rmse']:.3f}  MAPE={test_metrics['mape']:.2f}%")

    # ---- Train/Val fitted 결과 ----
    train_preds_inv, train_masks, train_metrics = _infer_and_metrics(tr_loader, Y_train, M_train)
    val_preds_inv,   val_masks,   val_metrics   = _infer_and_metrics(va_loader, Y_val,   M_val)

    print(f"[TRAIN] MAE={train_metrics['mae']:.3f}  RMSE={train_metrics['rmse']:.3f}  MAPE={train_metrics['mape']:.2f}%")
    print(f"[VAL]   MAE={val_metrics['mae']:.3f}  RMSE={val_metrics['rmse']:.3f}  MAPE={val_metrics['mape']:.2f}%")

    # 리턴 값 확장: (모델/학습곡선/셋별 지표와 예측/마스크/체크포인트 경로)
    return (model, train_losses, val_losses,
            test_metrics, test_preds_inv, test_masks, ckpt_path,
            train_preds_inv, train_masks, train_metrics,
            val_preds_inv,   val_masks,   val_metrics)


# =========================================================
# 4) 유틸: Dataset -> numpy 변환
# =========================================================
def ds_to_numpy(ds: DailyStepDataset):
    X_list, Y_list, M_list, y0_list = [], [], [], []
    for x, y, m, y0 in ds:
        X_list.append(x.numpy())
        Y_list.append(y.numpy())
        M_list.append(m.numpy())
        y0_list.append(float(y0.item()))
    X = np.stack(X_list)
    Y = np.stack(Y_list)
    M = np.stack(M_list)
    y0 = np.array(y0_list)
    return X, Y, M, y0


import random
from copy import deepcopy

def _sample_params(param_dist: dict) -> dict:
    """param_dist의 각 키에서 무작위 1개씩 뽑아 하이퍼파라미터 dict 생성"""
    params = {}
    for k, v in param_dist.items():
        if isinstance(v, (list, tuple)):
            params[k] = random.choice(v)
        else:
            params[k] = v
    return params

def random_search(
    X_train, Y_train, M_train, y0_train,
    X_val,   Y_val,   M_val,   y0_val,
    X_test,  Y_test,  M_test,  y0_test,
    scaler_y,
    param_dist: dict,
    n_trials: int = 10,
    device=None,
    max_epochs: int = 50,
    patience: int = 5,
    grad_clip: float = 1.0,
    score_metric: str = "rmse",  # "rmse" | "mae" | "mape"
    seed: int = 42,
):
    """
    검증 성능(기본: RMSE) 최소화 기준으로 랜덤서치 수행.
    trial마다 train_model()을 호출하고, best trial의 하이퍼파라미터/체크포인트/지표를 반환.
    """
    assert score_metric in {"rmse", "mae", "mape"}
    random.seed(seed)

    trials = []
    best_score = float("inf")
    best_params = None
    best_ckpt = None
    best_val_metrics = None
    best_test_metrics = None

    for t in range(1, n_trials + 1):
        params = _sample_params(param_dist)
        ckpt_path = f"rs_trial_{t:03d}.pth"

        print(f"\n[RandomSearch] Trial {t}/{n_trials}  params={params}")
        (_model, _tr_losses, _va_losses,
         test_metrics, _test_preds_inv, _test_masks, _ckpt_path,
         _train_preds_inv, _train_masks, train_metrics,
         _val_preds_inv,   _val_masks,   val_metrics) = train_model(
            X_train, Y_train, M_train, y0_train,
            X_val,   Y_val,   M_val,   y0_val,
            X_test,  Y_test,  M_test,  y0_test,
            scaler_y,
            params,
            device=device,
            max_epochs=max_epochs,
            patience=patience,
            grad_clip=grad_clip,
            ckpt_path=ckpt_path,
        )

        score = float(val_metrics[score_metric])
        trials.append({
            "trial": t,
            "params": deepcopy(params),
            "val_metrics": deepcopy(val_metrics),
            "test_metrics": deepcopy(test_metrics),
            "train_metrics": deepcopy(train_metrics),
            "score_metric": score_metric,
            "score": score,
            "ckpt_path": ckpt_path,
        })

        if score < best_score:
            best_score = score
            best_params = deepcopy(params)
            best_ckpt = ckpt_path
            best_val_metrics = deepcopy(val_metrics)
            best_test_metrics = deepcopy(test_metrics)

        print(f"[Trial {t}] val {score_metric.upper()} = {score:.4f}  "
              f"(best so far = {best_score:.4f})")

    trials.sort(key=lambda d: d["score"])
    return {
        "best_params": best_params,
        "best_score": best_score,
        "best_ckpt": best_ckpt,
        "best_val_metrics": best_val_metrics,
        "best_test_metrics": best_test_metrics,
        "score_metric": score_metric,
        "trials": trials,
    }

## 8. Dataset 구성 및 Scaler 적합

In [ ]:
from sklearn.preprocessing import StandardScaler

feature_cols = ["hm", "ta", "Fourier_Residual",
                "spring", "summer", "autoum", "winter", "is_holiday_dummies"]
target_col   = "Fourier_Residual"


scaler_X = StandardScaler().fit(train[feature_cols].values)
scaler_y = StandardScaler().fit(train[[target_col]].values)

train_ds = DailyStepDataset(
    train,
    feature_cols,
    target_col,
    in_days=7,        # 일 단위로만 구축
    out_days=1,       # 일 단위만 넣기
    scaler_X=scaler_X,
    scaler_y=scaler_y
)

val_ds = DailyStepDataset(
    val,
    feature_cols,
    target_col,
    in_days=7,        # 일 단위로만 구축
    out_days=1,       # 일 단위만 넣기
    scaler_X=scaler_X,
    scaler_y=scaler_y
)

test_ds = DailyStepDataset(
    test,
    feature_cols,
    target_col,
    in_days=7,        # 일 단위로만 구축
    out_days=1,       # 일 단위만 넣기
    scaler_X=scaler_X,
    scaler_y=scaler_y
)

## 9. Numpy 변환

In [ ]:
X_train, Y_train, M_train, y0_train = ds_to_numpy(train_ds)
X_val,   Y_val,   M_val,   y0_val   = ds_to_numpy(val_ds)
X_test,  Y_test,  M_test,  y0_test  = ds_to_numpy(test_ds)

print(f"X_train={X_train.shape}, Y_train={Y_train.shape}")
print(f"X_val={X_val.shape}, Y_val={Y_val.shape}")
print(f"X_test={X_test.shape}, Y_test={Y_test.shape}")

## 10. 랜덤서치 학습 실행

In [ ]:
device = TORCH_DEVICE
print(f"device: {device}")

param_dist = {
    "lstm_layers": [1, 2, 3],
    "units": [64, 128, 256],
    "learning_rate": [1e-3, 5e-4, 3e-4, 1e-4],
    "teacher_forcing_ratio": [0.3, 0.5, 0.7],
    "lstm_dropout_rate": [0.0, 0.1, 0.2],
    "batch_size": [32, 64, 128],
}

search_results = random_search(
    X_train, Y_train, M_train, y0_train,
    X_val,   Y_val,   M_val,   y0_val,
    X_test,  Y_test,  M_test,  y0_test,
    scaler_y,
    param_dist,
    n_trials=5,
    device=device,
    max_epochs=50,
    patience=5,
    grad_clip=1.0,
)

best_params = search_results["best_params"]
print(f"
최적 하이퍼파라미터: {best_params}")

(model, train_losses, val_losses,
 test_metrics, test_preds_inv, test_masks, ckpt_path,
 train_preds_inv, train_masks, train_metrics,
 val_preds_inv,   val_masks,   val_metrics) = train_model(
    X_train, Y_train, M_train, y0_train,
    X_val,   Y_val,   M_val,   y0_val,
    X_test,  Y_test,  M_test,  y0_test,
    scaler_y,
    best_params,
    device=device,
    max_epochs=150,
    patience=10,
    grad_clip=1.0,
)

## 11. 추론 결과 역스케일

In [ ]:
# train_model이 이미 역스케일된 결과를 반환하므로 직접 사용
train_pred_inv = train_preds_inv
val_pred_inv   = val_preds_inv
test_pred_inv  = test_preds_inv

print("✅ 학습 & 추론 완료")
print("Train preds:", train_pred_inv.shape)
print("Val preds:",   val_pred_inv.shape)
print("Test preds:",  test_pred_inv.shape)

## 12. 결과 데이터프레임 구성

In [ ]:
train1 = train[-len(train_pred_inv.flatten()):].copy()
train1["pred_inv"] = train_pred_inv.flatten()
train1["residual"] = train1["Fourier_Residual"] - train_pred_inv.flatten()

# 
val1 = val[-len(val_pred_inv.flatten()):].copy()
val1["pred_inv"] = val_pred_inv.flatten()
val1["residual"] = val1["Fourier_Residual"] - val_pred_inv.flatten()


test1 = test[-len(test_pred_inv.flatten()):].copy()
test1["pred_inv"] = test_pred_inv.flatten()
test1["residual"] = test1["Fourier_Residual"] - test_pred_inv.flatten()

## 13. 하이브리드 베이스라인

논문 Algorithm 1, Step A5

```
ŷ_t^base = T̂_t + Ŝ_t^(F) + r̂_t^(S2S)
```

In [ ]:
# 
train1["hybrid"] = train1["trend"] +train1["Fourier_Seasonality"] + train1["pred_inv"]
val1["hybrid"] =  val1["trend"] +val1["Fourier_Seasonality"] + val1["pred_inv"]
test1["hybrid"] =  test1["trend"] +test1["Fourier_Seasonality"] + test1["pred_inv"]

## 14. 베이스라인 평가

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error
real = test1["power demand(MW)"]
# 일반 평가지표 
print(f"MSE : {np.round(mean_squared_error(real, y_hat),5)}")
print(f"r2_score : {np.round(r2_score(real, y_hat),5)}")
print(f"MAPE : {np.round(mean_absolute_percentage_error(real, y_hat),5)}")
print(f"RMSE : {np.round(np.sqrt(mean_squared_error(real, y_hat)),5)}")

## 15. Hierarchical Quadratic Tilting (HQT)

논문 Algorithm 1, Step B–E

In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Tuple
# PyMC + LKJ
import pymc as pm
import pytensor.tensor as pt
from scipy.stats import norm


# =========================
# 0) 기본 설정 (추석/설날만)
# =========================


CHUSEOK_CORE = {"Chuseok"}
CHUSEOK_ALIASES = {
    "The day preceding Chuseok",
    "The second day of Chuseok",
    "Alternative holiday for Chuseok",
}
SEOLLAL_CORE = {"Korean New Year"}
SEOLLAL_ALIASES = {
    "The day preceding Korean New Year",
    "The second day of Korean New Year",
    "Alternative holiday for Korean New Year",
}

CHUSEOK_LABELS = CHUSEOK_CORE | CHUSEOK_ALIASES
SEOLLAL_LABELS = SEOLLAL_CORE | SEOLLAL_ALIASES


@dataclass
class HQTResult:
    # posterior draws
    draws_beta: np.ndarray         # [S, I, 3]  (이벤트 i별 β 샘플)
    draws_mu: np.ndarray           # [S, H, 3]  (유형별 μ 샘플; H=2: Chuseok/Seollal)
    draws_L: List[np.ndarray]      # [H] 각 (S, 3, 3): 사후 Cholesky factor L_h (논문 Σ_h = L_h L_h^T)
    draws_sigma_r: np.ndarray      # [S]
    # indices / metadata
    event_ids: List[str]           # 학습에 들어간 이벤트 ID 순서 (원본 정렬)
    event_type: List[str]          # 각 이벤트의 유형 ("Chuseok"|"Seollal")
    type_names: List[str]          # ["Chuseok","Seollal"] 고정
    tau_map: Dict[Tuple[str, "pd.Timestamp"], int]   # (event_id, ts) -> τ (정수, tau_unit 단위)
    event_id_of_date: Dict["pd.Timestamp", str]      # ts -> event_id (전체 기간)
    type_of_date: Dict["pd.Timestamp", str]          # ts -> "Chuseok"/"Seollal"
    # τ 메타
    tau_unit_hours: float          # τ의 기본 단위(시간) 예: 1.0 (1시간)
    tau_scale_hours: float         # 모형에서 τ를 나눌 스케일(시간) 예: 24.0 (하루)


def _sample_new_event_beta(
    rng: np.random.Generator,
    mu_draws_h: np.ndarray,   # (S, 3)  : 해당 타입 h의 μ 사후 드로우
    L_draws_h: np.ndarray,    # (S, 3, 3): 해당 타입 h의 Cholesky L_h 사후 드로우
) -> np.ndarray:
    """
    논문 수식: β_new ~ N(μ_h, Σ_h),  Σ_h = L_h L_h^T
    각 draw s에 대해 β_new,s = μ_{h,s} + L_{h,s} @ ε_s,  ε_s ~ N(0, I)
    → 사후 Cholesky factor L_h를 직접 사용 (표본공분산 근사 제거)
    반환: (S, 3)
    """
    S = mu_draws_h.shape[0]
    eps = rng.standard_normal((S, 3))                        # (S, 3)
    betas = mu_draws_h + np.einsum("sij,sj->si", L_draws_h, eps)  # (S, 3)
    return betas


# ==================================
# 1) 윈도우/τ 생성 (라벨 기반: radius 불필요)
# ==================================

def _date_index(df: pd.DataFrame, date_col: Optional[str]) -> pd.Series:
    if date_col is not None:
        idx = pd.to_datetime(df[date_col])
    else:
        if not isinstance(df.index, pd.DatetimeIndex):
            raise ValueError("date_col을 지정하지 않았고, 인덱스가 DatetimeIndex가 아닙니다.")
        idx = pd.to_datetime(df.index)
    return idx


def build_holiday_windows_and_tau_by_label(
    df_all: pd.DataFrame,
    holiday_name_col: str = "holiday_name",
    date_col: Optional[str] = None,
    center_hour: int = 0,
    tau_unit: str = "1h",
    pre_pad_days: int = 0,
    post_pad_days: int = 0,
) -> Tuple[Dict[str, List["pd.Timestamp"]],
           Dict[Tuple[str, "pd.Timestamp"], int],
           Dict["pd.Timestamp", str],
           Dict["pd.Timestamp", str],
           float]:
    idx = _date_index(df_all, date_col)
    names = df_all[holiday_name_col].astype(str).values
    df_tmp = pd.DataFrame({"date": idx, "holiday": names}).set_index("date")

    tau_unit_td = pd.to_timedelta(tau_unit)
    tau_unit_hours = tau_unit_td / pd.Timedelta(hours=1)

    windows: Dict[str, List] = {}
    tau_map: Dict = {}
    event_id_of_date: Dict = {}
    type_of_date: Dict = {}

    for d, name in zip(idx, names):
        if name in CHUSEOK_CORE:
            tname, labels = "Chuseok", CHUSEOK_LABELS
        elif name in SEOLLAL_CORE:
            tname, labels = "Seollal", SEOLLAL_LABELS
        else:
            continue

        y = int(pd.Timestamp(d).year)
        eid = f"{y}_{tname}"
        if eid in windows:
            continue

        d0 = pd.Timestamp(d).normalize() + pd.Timedelta(hours=center_hour)

        mask_year = (df_tmp.index.year == y)
        mask_lab = df_tmp["holiday"].isin(labels)
        stamps = sorted(df_tmp.index[mask_year & mask_lab].unique().tolist())
        if not stamps:
            stamps = [d0]

        if pre_pad_days > 0 or post_pad_days > 0:
            date_set = {pd.Timestamp(s).normalize() for s in stamps}
            min_date = min(date_set)
            max_date = max(date_set)
            start_pad_date = min_date - pd.Timedelta(days=pre_pad_days)
            end_pad_date   = max_date + pd.Timedelta(days=post_pad_days)
            base_dates = df_tmp.index.normalize()
            mask_pad = (base_dates >= start_pad_date) & (base_dates <= end_pad_date)
            pad_stamps = df_tmp.index[mask_pad].unique().tolist()
            stamps = sorted(set(stamps).union(pad_stamps))

        windows[eid] = stamps

        for dd in stamps:
            dd = pd.Timestamp(dd)
            delta = dd - d0
            tau_float = delta / tau_unit_td
            tau_int = int(np.rint(float(tau_float)))
            tau_map[(eid, dd)] = tau_int
            event_id_of_date[dd] = eid
            type_of_date[dd] = tname

    return windows, tau_map, event_id_of_date, type_of_date, float(tau_unit_hours)


# ==================================
# 2) 표준화 잔차 / σ (비명절일에서)
# ==================================

def compute_sigma_and_residuals(
    df: pd.DataFrame,
    y_col: str,
    pred_col: str,
    holiday_name_col: str = "holiday_name",
) -> Tuple[float, pd.Series]:
    y = pd.to_numeric(df[y_col])
    yhat = pd.to_numeric(df[pred_col])
    resid = y - yhat
    mask_non = (df[holiday_name_col].astype(str) == "non-event")
    sigma = float(resid[mask_non].std(ddof=1))
    if not np.isfinite(sigma) or sigma <= 0:
        raise ValueError("비명절(non-event) 표본에서 σ를 계산할 수 없습니다. 입력을 확인하세요.")
    return sigma, resid


# ==================================
# 3) PyMC(LKJ) 계층 이차 틸트 적합 (non-centered)
# ==================================

def fit_hqt_pymc_lkj(
    df_train: pd.DataFrame,
    sigma_resid: float,
    resid_train: pd.Series,
    tau_map: Dict,
    event_id_of_date_all: Dict,
    type_of_date_all: Dict,
    tau_scale_hours: float = 24.0,
    tau_unit_hours: float = 1.0,
    chains: int = 4,
    draws: int = 1000,
    tune: int = 1000,
    target_accept: float = 0.95,
    random_seed: int = 2025,
    date_col: Optional[str] = None,
    sampler: str = "nuts",
) -> HQTResult:
    """
    z_t ~ N(β_{i0} + β_{i1} τs + β_{i2} τs², σ_r²)
    τs = (τ * tau_unit_hours) / tau_scale_hours
    β_i = μ_{h(i)} + L_{h(i)} ε_i,  ε_i ~ N(0, I)   # non-centered
    L_{h} : LKJCholeskyCov로부터 얻은 (3×3) 하삼각 → pm.Deterministic으로 trace 저장
             → 새 이벤트 사후예측 시 β_new,s = μ_{h,s} + L_{h,s} @ ε_s 에 직접 사용
    """
    idx = _date_index(df_train, date_col)
    dates = [d for d in idx if d in event_id_of_date_all]
    if len(dates) == 0:
        raise ValueError("학습 세트에 추석/설날 윈도우 날짜가 없습니다.")

    ev_ids_sorted = sorted({event_id_of_date_all[d] for d in dates}, key=str)
    I = len(ev_ids_sorted)
    eid_to_int = {e: i for i, e in enumerate(ev_ids_sorted)}

    types_sorted = ["Chuseok", "Seollal"]
    H = len(types_sorted)
    type_to_int = {t: i for i, t in enumerate(types_sorted)}
    h_of_event = np.array([type_to_int[e.split("_")[1]] for e in ev_ids_sorted], dtype=int)

    z_vec = np.array([resid_train.loc[d] / sigma_resid for d in dates], dtype=float)
    tau_vec = np.array([float(tau_map[(event_id_of_date_all[d], d)]) for d in dates], dtype=float)
    tau_scaled = (tau_vec * float(tau_unit_hours)) / float(tau_scale_hours)
    e_idx_orig = np.array([eid_to_int[event_id_of_date_all[d]] for d in dates], dtype=int)

    idx_by_type = [np.where(h_of_event == h)[0] for h in range(H)]
    concat_order = np.concatenate([ix for ix in idx_by_type if len(ix) > 0])
    pos_of_event = np.empty(I, dtype=int)
    for pos, orig_idx in enumerate(concat_order):
        pos_of_event[orig_idx] = pos
    e_idx = pos_of_event[e_idx_orig]

    with pm.Model() as m:
        mu = pm.Normal("mu", mu=0.0, sigma=10.0, shape=(H, 3))

        # LKJCholeskyCov → L_h를 pm.Deterministic으로 trace에 저장
        # 논문: Σ_h ~ LKJ(2),  β_new ~ N(μ_h, Σ_h) 를 직접 구현하기 위해 필수
        L_list = []
        for h in range(H):
            sd = pm.HalfNormal.dist(1.0, shape=3)
            packed = pm.LKJCholeskyCov(
                f"chol_packed_{h}", n=3, eta=2.0, sd_dist=sd,
                compute_corr=False, store_in_trace=False
            )
            L_h_raw = pm.expand_packed_triangular(3, packed, lower=True)  # (3,3)
            # trace에 저장 → 새 이벤트 β_new 샘플링 시 직접 사용
            L_h = pm.Deterministic(f"L_chol_{h}", L_h_raw)
            L_list.append(L_h)

        beta_blocks = []
        for h in range(H):
            idx_h = idx_by_type[h]
            n_h = len(idx_h)
            if n_h > 0:
                eps = pm.Normal(f"eps_type{h}", mu=0.0, sigma=1.0, shape=(n_h, 3))
                beta_h = pm.Deterministic(
                    f"beta_type{h}", mu[h] + eps @ L_list[h].T
                )
                beta_blocks.append(beta_h)

        beta_concat = beta_blocks[0] if len(beta_blocks) == 1 else pt.concatenate(beta_blocks, axis=0)

        sigma_r = pm.HalfNormal("sigma_r", 1.0)

        mu_z = (beta_concat[e_idx, 0]
                + beta_concat[e_idx, 1] * tau_scaled
                + beta_concat[e_idx, 2] * pt.sqr(tau_scaled))
        pm.Normal("z_like", mu=mu_z, sigma=sigma_r, observed=z_vec)

        if sampler == "advi":
            approx = pm.fit(50_000, random_seed=random_seed)
            idata = approx.sample(draws=draws, random_seed=random_seed)

        elif sampler == "numpyro":
            try:
                import pymc.sampling.jax as pmjax
            except Exception as e:
                raise RuntimeError(
                    "sampler='numpyro'를 요청했지만 'pymc.sampling.jax'를 불러오지 못했습니다."
                ) from e
            _kwargs = dict(
                chains=chains, draws=draws, tune=tune,
                target_accept=target_accept, random_seed=random_seed,
                progressbar=False
            )
            try:
                idata = pmjax.sample_numpyro_nuts(
                    **_kwargs,
                    chain_method="vectorized",
                    postprocessing_backend="cpu",
                    idata_kwargs={"log_likelihood": False},
                )
            except TypeError:
                idata = pmjax.sample_numpyro_nuts(**_kwargs)

        else:  # "nuts"
            idata = pm.sample(
                chains=chains, draws=draws, tune=tune,
                target_accept=target_accept, random_seed=random_seed, progressbar=False
            )

    # posterior 정리 ─ beta
    arrays = []
    for h in range(H):
        key = f"beta_type{h}"
        if key in idata.posterior:
            arr = idata.posterior[key].values          # (chain, draw, n_h, 3)
            arr = np.moveaxis(arr, 0, 1).reshape(-1, arr.shape[2], 3)
            arrays.append(arr)
    draws_beta_concat = arrays[0] if len(arrays) == 1 else np.concatenate(arrays, axis=1)
    draws_beta = draws_beta_concat[:, pos_of_event, :]

    mu_draws = idata.posterior["mu"].values            # (chain, draw, H, 3)
    mu_draws = np.moveaxis(mu_draws, 0, 1).reshape(-1, H, 3)

    sigma_r_draws = idata.posterior["sigma_r"].values  # (chain, draw)
    sigma_r_draws = np.moveaxis(sigma_r_draws, 0, 1).reshape(-1)

    # posterior 정리 ─ Cholesky L_h  (논문 Σ_h = L_h L_h^T)
    draws_L: List[np.ndarray] = []
    for h in range(H):
        key = f"L_chol_{h}"
        if key in idata.posterior:
            arr = idata.posterior[key].values          # (chain, draw, 3, 3)
            arr = np.moveaxis(arr, 0, 1).reshape(-1, 3, 3)  # (S, 3, 3)
        else:
            # fallback: 단위행렬 (샘플링 실패 시 안전장치)
            S = mu_draws.shape[0]
            arr = np.tile(np.eye(3), (S, 1, 1))
        draws_L.append(arr)

    return HQTResult(
        draws_beta=draws_beta,
        draws_mu=mu_draws,
        draws_L=draws_L,
        draws_sigma_r=sigma_r_draws,
        event_ids=ev_ids_sorted,
        event_type=[types_sorted[h] for h in h_of_event],
        type_names=types_sorted,
        tau_map={k: int(v) for k, v in tau_map.items()},
        event_id_of_date=event_id_of_date_all.copy(),
        type_of_date=type_of_date_all.copy(),
        tau_unit_hours=float(tau_unit_hours),
        tau_scale_hours=float(tau_scale_hours),
    )


# ==================================
# 4) 틸트 계산/적용
# ==================================

def tilt_from_posterior_LKJ(
    hqt: HQTResult,
    sigma_resid: float,
    dates: Iterable["pd.Timestamp"],
    tilt_mode: str = "hybrid",
    ci: float = 0.95,
    rng_seed: Optional[int] = 2025,
) -> Tuple[pd.Series, float, pd.Series]:
    """
    새 이벤트 β_new 샘플링:
      논문: β_new ~ N(μ_h, Σ_h)
      구현: β_new,s = μ_{h,s} + L_{h,s} @ ε_s  (ε_s ~ N(0,I))
            사후 Cholesky L_{h,s} (draws_L)를 직접 사용
    """
    dates = list(pd.to_datetime(pd.Index(dates)))

    B_draws  = hqt.draws_beta   # (S, I, 3)
    MU_draws = hqt.draws_mu     # (S, H, 3)
    S = B_draws.shape[0]

    sigma_r_sq_mean = float((hqt.draws_sigma_r ** 2).mean())
    zcrit = norm.ppf(0.5 + ci / 2.0)
    half_width_const = float(zcrit * sigma_resid * np.sqrt(1.0 + sigma_r_sq_mean))

    eid_to_pos  = {e: i for i, e in enumerate(hqt.event_ids)}
    type_to_pos = {t: i for i, t in enumerate(hqt.type_names)}

    rng = np.random.default_rng(rng_seed)

    zhat_vals: List[float] = []
    hw_vals:   List[float] = []

    for d in dates:
        eid = hqt.event_id_of_date.get(d)
        if eid is None:
            zhat_vals.append(0.0)
            hw_vals.append(0.0)
            continue

        tau   = hqt.tau_map.get((eid, d), 0)
        tau_s = (float(tau) * hqt.tau_unit_hours) / hqt.tau_scale_hours

        if tilt_mode == "type":
            h = type_to_pos[hqt.type_of_date[d]]
            MU_h = MU_draws[:, h, :]
            z_draws = MU_h[:,0] + MU_h[:,1]*tau_s + MU_h[:,2]*(tau_s**2)

        elif tilt_mode == "event":
            if eid in eid_to_pos:
                i = eid_to_pos[eid]
                B_i = B_draws[:, i, :]
                z_draws = B_i[:,0] + B_i[:,1]*tau_s + B_i[:,2]*(tau_s**2)
            else:
                # 새 이벤트: 논문 β_new ~ N(μ_h, Σ_h) = μ_{h,s} + L_{h,s} @ ε_s
                h = type_to_pos[hqt.type_of_date[d]]
                betas_new = _sample_new_event_beta(
                    rng, MU_draws[:, h, :], hqt.draws_L[h]
                )
                z_draws = betas_new[:,0] + betas_new[:,1]*tau_s + betas_new[:,2]*(tau_s**2)

        else:  # hybrid
            if eid in eid_to_pos:
                i = eid_to_pos[eid]
                B_i = B_draws[:, i, :]
                z_draws = B_i[:,0] + B_i[:,1]*tau_s + B_i[:,2]*(tau_s**2)
            else:
                # 새 이벤트: 동일하게 사후 L_h 사용
                h = type_to_pos[hqt.type_of_date[d]]
                betas_new = _sample_new_event_beta(
                    rng, MU_draws[:, h, :], hqt.draws_L[h]
                )
                z_draws = betas_new[:,0] + betas_new[:,1]*tau_s + betas_new[:,2]*(tau_s**2)

        z_mean = float(np.mean(z_draws))
        std_z  = float(np.std(z_draws, ddof=1))
        zhat_vals.append(z_mean)

        half_width_t = float(zcrit * np.sqrt(
            (sigma_resid**2) * (1.0 + sigma_r_sq_mean + std_z**2)
        ))
        hw_vals.append(half_width_t)

    zhat         = pd.Series(zhat_vals, index=pd.Index(dates, name="date"), name="z_hat")
    e_tilt       = sigma_resid * zhat
    half_width_t = pd.Series(hw_vals, index=zhat.index, name="half_width_t")
    return e_tilt, half_width_const, half_width_t


def apply_tilt(baseline_pred: pd.Series, e_tilt: pd.Series) -> pd.Series:
    return (baseline_pred + e_tilt.reindex(baseline_pred.index).fillna(0.0)).rename("tilted_pred")


# ==================================
# 5) 지표
# ==================================

def mae(y, yhat): return float(np.mean(np.abs(np.asarray(y) - np.asarray(yhat))))
def rmse(y, yhat): return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(yhat)) ** 2)))

def trough_bias_per_event(
    y: pd.Series, pred: pd.Series, windows: Dict[str, List]
) -> float:
    biases = []
    for eid, days in windows.items():
        ds = [d for d in days if d in y.index]
        if not ds:
            continue
        dmin = y.loc[ds].idxmin()
        biases.append(float(pred.loc[dmin] - y.loc[dmin]))
    return float(np.mean(biases)) if biases else np.nan


def interval_stats(y: pd.Series, pred: pd.Series, half_width: float):
    lower = pred - half_width
    upper = pred + half_width
    picp = float(np.mean((y >= lower) & (y <= upper)) * 100.0)
    aiw  = float(np.mean(upper - lower))
    return picp, aiw


# ==================================
# 6) End-to-End 헬퍼
# ==================================

def run_hqt_pipeline_LKJ(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    y_col: str,
    pred_col: str,
    holiday_name_col: str = "holiday_name",
    date_col: Optional[str] = None,
    tilt_mode: str = "hybrid",
    chains: int = 4,
    draws: int = 1000,
    tune: int = 1000,
    target_accept: float = 0.95,
    sampler: str = "nuts",
    random_seed: int = 2025,
    center_hour: int = 0,
    tau_unit: str = "1h",
    tau_scale_hours: float = 24.0,
    pre_pad_days: int = 1,
    post_pad_days: int = 1,
):
    all_df = pd.concat(
        [train_df[[holiday_name_col]], val_df[[holiday_name_col]], test_df[[holiday_name_col]]]
    )
    windows, tau_map, eid_of_date, type_of_date, tau_unit_hours = build_holiday_windows_and_tau_by_label(
        all_df, holiday_name_col=holiday_name_col, date_col=date_col,
        center_hour=center_hour, tau_unit=tau_unit,
        pre_pad_days=pre_pad_days, post_pad_days=post_pad_days,
    )

    sigma_resid, resid_train = compute_sigma_and_residuals(train_df, y_col, pred_col, holiday_name_col)

    hqt = fit_hqt_pymc_lkj(
        df_train=train_df,
        sigma_resid=sigma_resid,
        resid_train=resid_train,
        tau_map=tau_map,
        event_id_of_date_all=eid_of_date,
        type_of_date_all=type_of_date,
        tau_scale_hours=tau_scale_hours,
        tau_unit_hours=tau_unit_hours,
        chains=chains,
        draws=draws,
        tune=tune,
        target_accept=target_accept,
        sampler=sampler,
        random_seed=random_seed,
        date_col=date_col,
    )

    def _one_split(df):
        idx  = _date_index(df, date_col)
        y    = pd.to_numeric(df[y_col])
        base = pd.to_numeric(df[pred_col]).rename("baseline_pred")

        e_tilt, half_width_const, half_width_t = tilt_from_posterior_LKJ(
            hqt, sigma_resid, idx, tilt_mode=tilt_mode
        )
        tilted = apply_tilt(base, e_tilt)

        holiday_ts = [d for d in idx if d in eid_of_date]
        mask_h = pd.Index(idx).isin(holiday_ts)

        lower_t = tilted - half_width_t.reindex(idx).fillna(0.0)
        upper_t = tilted + half_width_t.reindex(idx).fillna(0.0)

        if np.any(mask_h):
            picp = float(np.mean((y[mask_h] >= lower_t[mask_h]) & (y[mask_h] <= upper_t[mask_h])) * 100.0)
            aiw  = float(np.mean((upper_t - lower_t)[mask_h]))
        else:
            picp, aiw = np.nan, np.nan

        metrics = {
            "MAE_all":      mae(y, tilted),
            "RMSE_all":     rmse(y, tilted),
            "MAE_holiday":  mae(y[mask_h], tilted[mask_h]) if np.any(mask_h) else np.nan,
            "RMSE_holiday": rmse(y[mask_h], tilted[mask_h]) if np.any(mask_h) else np.nan,
            "PICP_95":      picp,
            "AIW":          aiw,
            "half_width_const": half_width_const,
        }

        out = pd.DataFrame({
            "y":            y,
            "baseline_pred": base,
            "e_tilt":       e_tilt.reindex(idx).fillna(0.0),
            "tilted_pred":  tilted,
            "half_width_t": half_width_t.reindex(idx).fillna(0.0),
            "lower_t":      lower_t,
            "upper_t":      upper_t,
        }, index=idx)
        return out, metrics

    train_out, train_metrics = _one_split(train_df)
    val_out,   val_metrics   = _one_split(val_df)
    test_out,  test_metrics  = _one_split(test_df)

    return {
        "hqt":          hqt,
        "sigma_resid":  sigma_resid,
        "windows":      windows,
        "train":        {"preds": train_out, "metrics": train_metrics},
        "val":          {"preds": val_out,   "metrics": val_metrics},
        "test":         {"preds": test_out,  "metrics": test_metrics},
    }


## 16. 인덱스 설정 (DatetimeIndex)

In [ ]:
# 최초 한번만 처리

train1.set_index("일시", inplace=True)
val1.set_index("일시", inplace=True)
test1.set_index("일시", inplace=True)

## 17. HQT 파이프라인 실행

In [ ]:
results_event = run_hqt_pipeline_LKJ(
    train_df=train1,
    val_df=val1,
    test_df=test1,
    
    y_col="power demand(MW)",
    
    pred_col="hybrid",
    
    holiday_name_col="holiday_name",
    
    date_col=None,             
    sampler=MCMC_SAMPLER,                    # "nuts"|"numpyro"|"advi"
    target_accept=0.99,                # 안정화 권장
    chains=4, draws=3000, tune=2000,
    tilt_mode="hybrid",   # "event" | "type" | "hybrid"
    pre_pad_days=1,                # ← 시작 하루 전 포함
    post_pad_days=1,               # ← 종료 하루 후 포함
)

## 18. 결과 확인 및 시각화

In [ ]:
hybrid_tilt = results_event["test"]["preds"]
hybrid_tilt_1 = hybrid_tilt[hybrid_tilt["e_tilt"] != 0]
hybrid_tilt_1

In [ ]:
# y	baseline_pred	e_tilt	tilted_pred
plt.plot(hybrid_tilt_1["y"][24:24*6], label = "real")
plt.plot(hybrid_tilt_1["tilted_pred"][24:24*6], label = "tilt")
plt.plot(hybrid_tilt_1["baseline_pred"][24:24*6], label = "hybrid")

plt.legend()

In [ ]:
# y	baseline_pred	e_tilt	tilted_pred
plt.plot(hybrid_tilt_1["y"][24*6:], label = "real")
plt.plot(hybrid_tilt_1["tilted_pred"][24*6:], label = "tilt")
plt.plot(hybrid_tilt_1["baseline_pred"][24*6:], label = "hybrid")

plt.legend()

## 20. 최종 평가 지표

In [ ]:
# 전체 결과에 대한 평가지표 생성
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, r2_score

y_true = hybrid_tilt["y"].values
y_pred_tilt = hybrid_tilt["tilted_pred"].values
y_pred = hybrid_tilt["baseline_pred"].values

print(f"MSE : {np.round(mean_squared_error(y_true, y_pred_tilt), 5)}")
print(f"r2_score : {np.round(r2_score(y_true, y_pred_tilt), 5)}")
print(f"MAPE : {np.round(mean_absolute_percentage_error(y_true, y_pred_tilt), 5)}")
print(f"RMSE : {np.round(np.sqrt(mean_squared_error(y_true, y_pred_tilt)), 5)}")

print("위는 틸팅 전 ======================== 아래는 틸팅 후\n")

print(f"MSE : {np.round(mean_squared_error(y_true, y_pred), 5)}")
print(f"r2_score : {np.round(r2_score(y_true, y_pred), 5)}")
print(f"MAPE : {np.round(mean_absolute_percentage_error(y_true, y_pred), 5)}")
print(f"RMSE : {np.round(np.sqrt(mean_squared_error(y_true, y_pred)), 5)}")

In [ ]:

y_true1 = hybrid_tilt_1["y"].values
y_pred_tilt1 = hybrid_tilt_1["tilted_pred"].values
y_pred1 = hybrid_tilt_1["baseline_pred"].values

In [ ]:
print(f"MSE : {np.round(mean_squared_error(y_true1, y_pred_tilt1), 5)}")
print(f"r2_score : {np.round(r2_score(y_true1, y_pred_tilt1), 5)}")
print(f"MAPE : {np.round(mean_absolute_percentage_error(y_true1, y_pred_tilt1), 5)}")

print("위는 틸팅 전 ======================== 아래는 틸팅 후")
print(f"MSE : {np.round(mean_squared_error(y_true1, y_pred1), 5)}")
print(f"r2_score : {np.round(r2_score(y_true1,  y_pred1), 5)}")
print(f"MAPE : {np.round(mean_absolute_percentage_error(y_true1,  y_pred1), 5)}")
